# Notebook 01 -- Data Pipeline & Baseline RAG

**Config 1 (Baseline):** multilingual-e5-large + BM25 + RRF fusion + Qwen2.5-7B-Instruct (4-bit)

Designed for Google Colab (T4/A100). Every section checks if its output
already exists on Drive before running, so after a kernel crash you just
re-run the Setup cell and skip ahead.

**OOM fix:** The previous version crashed on the 702k-row Yargitay dataset
because it loaded everything into pandas. This version skips the "build corpus"
step entirely and goes **raw parquet -> chunks** using `chunk_parquet_streaming`,
which reads in 10k-row batches and never holds more than one batch in RAM.

| Section | Output on Drive |
|---|---|
| 2. Download | `data/raw/*.parquet` |
| 3. Chunks | `data/processed/chunks.parquet` |
| 4. FAISS | `indexes/faiss.index` |
| 5. BM25 | `indexes/bm25.pkl` |
| 9-10. Eval | `results/baseline_config1.json` |

## 1. Setup

In [ ]:
# ====================================================================
# FILL IN THESE 3 VALUES AND RUN THIS CELL FIRST (AFTER EVERY RESTART)
# ====================================================================
GITHUB_TOKEN    = ""   # github.com/settings/tokens -> repo scope
KAGGLE_USERNAME = ""   # kaggle.com -> Account -> API
KAGGLE_KEY      = ""   # kaggle.com -> Account -> API
# ====================================================================

# ---- Drive mount ----
from google.colab import drive
drive.mount("/content/drive")

# ---- Paths (hardcoded, no config.yaml dependency) ----
from pathlib import Path
import os, sys, json, logging, gc

DRIVE_ROOT   = Path("/content/drive/MyDrive/hukuk-rag")
RAW_DIR      = DRIVE_ROOT / "data" / "raw"
CHUNKS_PATH  = DRIVE_ROOT / "data" / "processed" / "chunks.parquet"
FAISS_PATH   = DRIVE_ROOT / "indexes" / "faiss.index"
BM25_PATH    = DRIVE_ROOT / "indexes" / "bm25.pkl"
RESULTS_DIR  = DRIVE_ROOT / "results"
GOLD_PATH    = Path("/content/hukuk-rag/data/gold/gold_test_set.json")
GOLD_TEMPLATE = Path("/content/hukuk-rag/data/gold/gold_test_template.json")
PROJECT_ROOT = Path("/content/hukuk-rag")

for d in [RAW_DIR, CHUNKS_PATH.parent, FAISS_PATH.parent, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- Kaggle credentials (non-fatal) ----
try:
    os.makedirs("/root/.kaggle", exist_ok=True)
    with open("/root/.kaggle/kaggle.json", "w") as _f:
        json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, _f)
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
except Exception as e:
    print(f"[WARNING] Kaggle creds setup failed: {e}")

# ---- Clone / pull repo ----
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/berkay-aktas/hukuk-rag.git"
if not PROJECT_ROOT.exists():
    !git clone $REPO_URL $PROJECT_ROOT
else:
    !cd $PROJECT_ROOT && git pull

# ---- Install dependencies ----
!pip install -q -r {PROJECT_ROOT / 'requirements.txt'}

# ---- sys.path for src.* imports ----
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---- Logging ----
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(name)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

# ---- Seeds & device ----
import random, numpy as np, torch
for s in [random.seed, np.random.seed, torch.manual_seed]:
    s(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Drive root: {DRIVE_ROOT}")
print("Setup complete.")

## 2. Download Datasets

- **HuggingFace** (3 datasets): Yargitay 700k, Turkish Law QA, Turkish Law Chatbot
- **Kaggle** (1 dataset): Turkish Law for LLM Fine-tuning

Each download is skipped if its Parquet file already exists on Drive.
After each download the dataset object is deleted and garbage collected.

In [ ]:
from datasets import load_dataset

HF_DATASETS = [
    {"id": "erdem-erdem/Turkish-Law-Documents-700k-clustered", "name": "yargitay_700k"},
    {"id": "OrionCAF/turkish_law_qa_dataset",                 "name": "turkish_law_qa"},
    {"id": "Renicames/turkish-law-chatbot",                   "name": "turkish_law_chatbot"},
]

KAGGLE_SLUG = "batuhankalem/turkishlaw-dataset-for-llm-finetuning"
KAGGLE_NAME = "kaggle_turkish_law"

# ---- HuggingFace downloads ----
for ds_info in HF_DATASETS:
    out_path = RAW_DIR / f"{ds_info['name']}.parquet"
    if out_path.exists():
        print(f"[skip] {ds_info['name']} already at {out_path}")
        continue
    print(f"Downloading {ds_info['id']}...")
    ds = load_dataset(ds_info["id"], split="train")
    ds.to_parquet(str(out_path))
    print(f"  Saved {ds_info['name']} ({len(ds):,} rows) -> {out_path}")
    del ds; gc.collect()

# ---- Kaggle download (non-fatal) ----
kaggle_dir = RAW_DIR / KAGGLE_NAME
if kaggle_dir.exists() and any(kaggle_dir.iterdir()):
    print(f"[skip] {KAGGLE_NAME} already at {kaggle_dir}")
else:
    try:
        import subprocess
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", KAGGLE_SLUG,
             "-p", str(kaggle_dir), "--unzip"],
            check=True,
        )
        print(f"  Saved {KAGGLE_NAME} -> {kaggle_dir}")
    except Exception as e:
        print(f"[WARNING] Kaggle download failed: {e}")
        print("  Continuing without Kaggle dataset.")

print("\nDownloads done.")

## 3. Chunk Corpus (STREAMING -- NO OOM)

**This is the cell that was crashing before.** The old version loaded all 702k
Yargitay rows into a single pandas DataFrame, which exceeded Colab's 12 GB RAM.

The fix: `chunk_parquet_streaming` reads each raw parquet file in **10k-row
batches**, chunks each batch using the legal-aware chunker, and writes results
incrementally to a new parquet file via PyArrow's `ParquetWriter`. At no point
are more than 10k rows in memory.

In [ ]:
import pandas as pd
from src.data.chunker import chunk_parquet_streaming

if CHUNKS_PATH.exists():
    print(f"[skip] Chunks already exist at {CHUNKS_PATH}")
else:
    # Build input list from whatever raw files actually exist
    RAW_SOURCES = [
        (RAW_DIR / "yargitay_700k.parquet",      "yargitay_700k"),
        (RAW_DIR / "turkish_law_qa.parquet",      "turkish_law_qa"),
        (RAW_DIR / "turkish_law_chatbot.parquet", "turkish_law_chatbot"),
    ]

    # Include any Kaggle parquet files if present
    kaggle_dir = RAW_DIR / "kaggle_turkish_law"
    if kaggle_dir.exists():
        for pf in sorted(kaggle_dir.glob("*.parquet")):
            RAW_SOURCES.append((pf, "kaggle_turkish_law"))

    input_paths = [(p, name) for p, name in RAW_SOURCES if p.exists()]
    print(f"Streaming chunker inputs ({len(input_paths)} files):")
    for p, name in input_paths:
        print(f"  {name}: {p}")

    total_chunks = chunk_parquet_streaming(
        input_paths=input_paths,
        output_path=CHUNKS_PATH,
        text_column="text",
        max_tokens=480,
        overlap_tokens=64,
        batch_size=10_000,
    )
    print(f"\nTotal chunks created: {total_chunks:,}")

# ---- Print stats from a sample of the output ----
import pyarrow.parquet as pq

pf = pq.ParquetFile(str(CHUNKS_PATH))
total_rows = pf.metadata.num_rows
print(f"\n--- Chunk Statistics ---")
print(f"Total chunks on disk: {total_rows:,}")

# Read a small sample for word-count stats
sample = next(pf.iter_batches(batch_size=min(5000, total_rows))).to_pandas()
wc = sample["text"].str.split().str.len()
print(f"Sample word counts (n={len(sample):,}):")
print(f"  Mean: {wc.mean():.1f}, Median: {wc.median():.1f}, Min: {wc.min()}, Max: {wc.max()}")
if "_source" in sample.columns:
    print(f"  Sources in sample: {sample['_source'].value_counts().to_dict()}")
del sample, wc; gc.collect()

## 3.5 Filter Short Chunks

The streaming chunker produced ~10M chunks. Many are very short (single sentences) which add noise and make indexing slow. Filter to chunks with >= 30 words. Also done in streaming batches.

In [ ]:
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd

FILTERED_PATH = DRIVE_ROOT / "data" / "processed" / "chunks_filtered.parquet"

if FILTERED_PATH.exists():
    print(f"[skip] Filtered chunks already exist at {FILTERED_PATH}")
else:
    print("Filtering short chunks (< 30 words)...")
    pf = pq.ParquetFile(str(CHUNKS_PATH))
    writer = None
    total_in, total_out = 0, 0

    for batch in pf.iter_batches(batch_size=100_000):
        df = batch.to_pandas()
        total_in += len(df)
        df = df[df["text"].str.split().str.len() >= 30]
        total_out += len(df)
        if len(df) > 0:
            table = pa.Table.from_pandas(df)
            if writer is None:
                writer = pq.ParquetWriter(str(FILTERED_PATH), table.schema)
            writer.write_table(table)
        del df; gc.collect()
        print(f"  Processed {total_in:,} -> kept {total_out:,}", end="")

    if writer:
        writer.close()
    print(f"
Filtered: {total_in:,} -> {total_out:,} chunks")

# Use filtered chunks for all downstream steps
CHUNKS_PATH = FILTERED_PATH
pf = pq.ParquetFile(str(CHUNKS_PATH))
print(f"Using {pf.metadata.num_rows:,} chunks for indexing")

## 4. Build FAISS Index

Uses `intfloat/multilingual-e5-large` embeddings with IVF-PQ quantization.
The embedding model is freed from GPU after indexing to make room for the LLM.

In [ ]:
import pandas as pd
from src.retrieval.dense import build_faiss_index, load_faiss_index

EMBEDDING_MODEL = "intfloat/multilingual-e5-large"

if FAISS_PATH.exists():
    print(f"[skip] FAISS index already exists at {FAISS_PATH}")
    faiss_index, faiss_mapping = load_faiss_index(FAISS_PATH)
else:
    # Load chunks -- these are the chunked output, much smaller than raw docs
    print("Loading chunks from parquet...")
    chunks_df = pd.read_parquet(CHUNKS_PATH)
    chunk_records = chunks_df.to_dict(orient="records")
    print(f"Loaded {len(chunk_records):,} chunks")
    del chunks_df; gc.collect()

    faiss_index, faiss_mapping = build_faiss_index(
        chunks=chunk_records,
        model_name=EMBEDDING_MODEL,
        save_path=FAISS_PATH,
        batch_size=64,
        nlist=256,
        m=32,
        nbits=8,
    )
    del chunk_records; gc.collect()

print(f"FAISS index: {faiss_index.ntotal:,} vectors")

## 5. Build BM25 Index

In [ ]:
import pandas as pd
from src.retrieval.bm25 import build_bm25_index, load_bm25_index

if BM25_PATH.exists():
    print(f"[skip] BM25 index already exists at {BM25_PATH}")
    bm25_index, bm25_mapping = load_bm25_index(BM25_PATH)
else:
    print("Loading chunks from parquet...")
    chunks_df = pd.read_parquet(CHUNKS_PATH)
    chunk_records = chunks_df.to_dict(orient="records")
    print(f"Loaded {len(chunk_records):,} chunks")

    bm25_index, bm25_mapping = build_bm25_index(
        chunks=chunk_records,
        save_path=BM25_PATH,
    )

    del chunk_records, chunks_df; gc.collect()

print(f"BM25 index: {len(bm25_mapping):,} chunks")

## 6. Sanity Check

Run a sample Turkish legal query against both indexes to verify they work.

In [ ]:
from src.retrieval.dense import dense_search, load_embedding_model
from src.retrieval.bm25 import bm25_search

SAMPLE_QUERY = "Kasten adam oldurme sucunun cezasi nedir?"

# Load embedding model for dense search
embed_model = load_embedding_model(EMBEDDING_MODEL)

dense_results = dense_search(
    query=SAMPLE_QUERY,
    index=faiss_index,
    chunk_mapping=faiss_mapping,
    model=embed_model,
    k=3,
    nprobe=16,
)

bm25_results = bm25_search(
    query=SAMPLE_QUERY,
    index=bm25_index,
    chunk_mapping=bm25_mapping,
    k=3,
)

print("=== Dense (FAISS) Top-3 ===")
for i, r in enumerate(dense_results, 1):
    print(f"\n[{i}] score={r.score:.4f}")
    print(f"    {r.text[:200]}...")

print("\n=== BM25 Top-3 ===")
for i, r in enumerate(bm25_results, 1):
    print(f"\n[{i}] score={r.score:.4f}")
    print(f"    {r.text[:200]}...")

## 7. Baseline RAG Pipeline

Load Qwen2.5-7B-Instruct in 4-bit (NF4 + double quantization).
Then reload the embedding model so both coexist in memory for inference.

In [ ]:
# Free embedding model first to make room for the LLM
del embed_model
gc.collect()
torch.cuda.empty_cache()
print("Embedding model freed from GPU.")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {LLM_NAME} in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
llm_model.eval()
print(f"LLM loaded. GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# Reload embedding model -- both LLM and embedder need to coexist for RAG
from src.retrieval.dense import load_embedding_model, dense_search
from src.retrieval.bm25 import bm25_search
from src.retrieval.fusion import rrf_merge

embed_model = load_embedding_model(EMBEDDING_MODEL)
print(f"Embedding model reloaded. GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
SYSTEM_PROMPT = (
    "Sen bir Turk hukuku uzmanisin. Sana verilen baglam paragraflarini kullanarak "
    "soruyu yanitla. Yanitinda ilgili kanun maddelerine atifta bulun. "
    "Eger baglam bilgisi yeterli degilse, bunu acikca belirt ve "
    "bilmedigin konularda uydurma yapma."
)


def format_context(results, top_k=10):
    """Format retrieval results into a numbered context string."""
    parts = []
    for i, r in enumerate(results[:top_k], 1):
        parts.append(f"[{i}] {r.text}")
    return "\n\n".join(parts)


def build_prompt(question, context_str):
    """Build chat-formatted prompt for Qwen."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Baglam:\n{context_str}\n\n"
                f"Soru: {question}\n\n"
                "Lutfen yukaridaki baglami kullanarak soruyu yanitla. "
                "Hangi kaynaklardan ([1], [2], ...) yararlandigini belirt."
            ),
        },
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


@torch.inference_mode()
def generate_answer(prompt, max_new_tokens=512, temperature=0.1, top_p=0.9):
    """Generate an answer from the LLM given a formatted prompt."""
    inputs = tokenizer(prompt, return_tensors="pt").to(llm_model.device)
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def baseline_rag(question):
    """Run the full baseline RAG pipeline for a single question.

    Steps: dense search -> BM25 search -> RRF merge -> generate answer.

    Returns:
        Tuple of (answer_text, merged_results).
    """
    d_results = dense_search(
        query=question,
        index=faiss_index,
        chunk_mapping=faiss_mapping,
        model=embed_model,
        k=50,
        nprobe=16,
    )

    b_results = bm25_search(
        query=question,
        index=bm25_index,
        chunk_mapping=bm25_mapping,
        k=50,
    )

    merged = rrf_merge(d_results, b_results, k=60, top_k=10)

    context_str = format_context(merged, top_k=10)
    prompt = build_prompt(question, context_str)
    answer = generate_answer(prompt)

    return answer, merged


print("RAG pipeline defined.")

## 8. Test with Examples

In [ ]:
TEST_QUESTIONS = [
    "Kasten adam oldurme sucunun cezasi nedir?",
    "Kira sozlesmesinde kiracinin haklari nelerdir?",
    "Idari yargida dava acma suresi ne kadardir?",
]

for q in TEST_QUESTIONS:
    print(f"\n{'=' * 80}")
    print(f"SORU: {q}")
    print("=" * 80)
    answer, results = baseline_rag(q)
    print(f"\nYANIT:\n{answer}")
    print(f"\nRetrieved {len(results)} passages (top-3 IDs):")
    for r in results[:3]:
        print(f"  - {r.chunk_id} (score={r.score:.4f})")

## 9. Evaluate on Gold Set

In [ ]:
from tqdm.auto import tqdm
from src.data.gold_set import load_gold_set, gold_set_stats
from src.evaluation.metrics import retrieval_metrics, generation_metrics

# Use full gold set if available, otherwise fall back to template
gold_path = GOLD_PATH if GOLD_PATH.exists() else GOLD_TEMPLATE
print(f"Loading gold set from: {gold_path}")

gold_data = load_gold_set(gold_path)
stats = gold_set_stats(gold_data)
print(f"Gold set: {stats['total']} questions")
print(f"  Domains: {stats['by_domain']}")
print(f"  Difficulties: {stats['by_difficulty']}")
print(f"  Answerable: {stats['answerable']}, Unanswerable: {stats['unanswerable']}")

In [ ]:
predictions = []
references = []
all_retrieved_ids = []
all_relevant_ids = []

for item in tqdm(gold_data, desc="Evaluating"):
    question = item["question"]
    gold_answer = item["gold_answer"]
    relevant_ids = item.get("relevant_doc_ids", [])

    answer, results = baseline_rag(question)

    predictions.append(answer)
    references.append(gold_answer)
    all_retrieved_ids.append([r.chunk_id for r in results])
    all_relevant_ids.append(relevant_ids)

print(f"Evaluated {len(predictions)} questions.")

In [ ]:
# ---- Retrieval metrics ----
has_relevant = [i for i, ids in enumerate(all_relevant_ids) if len(ids) > 0]

if has_relevant:
    filtered_retrieved = [all_retrieved_ids[i] for i in has_relevant]
    filtered_relevant = [all_relevant_ids[i] for i in has_relevant]
    ret_metrics = retrieval_metrics(filtered_retrieved, filtered_relevant)
else:
    print("No relevant_doc_ids in gold set -- retrieval metrics use empty baselines.")
    print("Populate relevant_doc_ids after indexing to get meaningful retrieval scores.")
    ret_metrics = {"recall@5": None, "recall@10": None, "mrr": None, "ndcg@10": None}

print("\n--- Retrieval Metrics ---")
for k, v in ret_metrics.items():
    print(f"  {k}: {v if v is None else f'{v:.4f}'}")

# ---- Generation metrics ----
gen_metrics = generation_metrics(predictions, references)

print("\n--- Generation Metrics ---")
for k, v in gen_metrics.items():
    print(f"  {k}: {v:.4f}")

## 10. Results Summary

In [ ]:
import json
import pandas as pd

baseline_results = {
    "config": "Config 1 -- Baseline",
    "embedding_model": EMBEDDING_MODEL,
    "llm_model": LLM_NAME,
    "quantization": "4-bit NF4 double-quant",
    "retrieval": {
        "method": "Dense (FAISS IVF-PQ) + BM25 + RRF",
        "dense_top_k": 50,
        "bm25_top_k": 50,
        "final_top_k": 10,
        "rrf_k": 60,
    },
    "chunking": {
        "max_tokens": 480,
        "overlap_tokens": 64,
        "method": "legal-aware streaming (10k batch, Madde/section/paragraph boundaries)",
    },
    "metrics": {
        "retrieval": {k: float(v) if v is not None else None for k, v in ret_metrics.items()},
        "generation": {k: float(v) for k, v in gen_metrics.items()},
    },
    "gold_set_size": len(gold_data),
}

# Display as table
display_rows = []
for category in ["retrieval", "generation"]:
    for metric, value in baseline_results["metrics"][category].items():
        display_rows.append({
            "Category": category.capitalize(),
            "Metric": metric,
            "Score": f"{value:.4f}" if value is not None else "N/A",
        })

results_table = pd.DataFrame(display_rows)
print("\n" + "=" * 50)
print("  CONFIG 1 (BASELINE) -- RESULTS")
print("=" * 50)
print(results_table.to_string(index=False))
print("=" * 50)

# Save to Drive
results_path = RESULTS_DIR / "baseline_config1.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, ensure_ascii=False, indent=2)
print(f"\nResults saved to {results_path}")

## 11. Cleanup

In [ ]:
del llm_model, tokenizer, embed_model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("Notebook complete.")